In [2]:
import cv2
import json
import os
from pathlib import Path

# ── 配置 ──────────────────────────────────────────────


          

VIDEO_DIR  = r'D:\EndoTracker\selected_dataset'
ANN_DIR    = r'D:\EndoTracker\vis_output\annotation'

CKPT_PATH  = r'D:\EndoTracker\checkpoints\reference_model\model-000600000.pth'
MAX_FRAMES = 100
N_POINTS=5
# ── 找所有视频 ────────────────────────────────────────
video_files = sorted(Path(VIDEO_DIR).glob("*.mp4"))
print(f"共找到 {len(video_files)} 个视频:")
for v in video_files:
    print(f"  {v.name}")

# ── 标注函数 ──────────────────────────────────────────
def click_event(event, x, y, flags, params):
    if event == cv2.EVENT_LBUTTONDOWN:
        params.append([x, y])
        print(f"  点 {len(params)}: ({x}, {y})")

def annotate_video(video_path, ann_path, n_points=5):
    seq_name = Path(video_path).stem
    
    # 已有标注则跳过
    if Path(ann_path).exists():
        print(f"\n[跳过] {seq_name}，标注文件已存在: {ann_path}")
        return

    print(f"\n{'='*50}")
    print(f"标注: {seq_name}")

    # 读视频
    cap = cv2.VideoCapture(str(video_path))
    frames = []
    while len(frames) < MAX_FRAMES:
        ret, frame = cap.read()
        if not ret: break
        frames.append(frame)
    cap.release()

    if len(frames) == 0:
        print(f"  [错误] 无法读取视频，跳过")
        return

    first_frame = frames[0]
    last_frame  = frames[-1]
    print(f"  共 {len(frames)} 帧，请标注第0帧和第{len(frames)-1}帧")

    points = {"f0": [], "fLast": [], "total_frames": len(frames)}

    # ── 标注第一帧 ────────────────────────────────────
    print(f"\n  >>> 第一帧：请点击 {n_points} 个点，完成后按任意键确认")
    cv2.imshow(f"[{seq_name}] First Frame - click {n_points} points, press any key", first_frame)
    cv2.setMouseCallback(f"[{seq_name}] First Frame - click {n_points} points, press any key",
                         click_event, points["f0"])
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    print(f"  第一帧标注完成，共 {len(points['f0'])} 个点")

    # ── 标注最后一帧 ──────────────────────────────────
    print(f"\n  >>> 最后一帧：请点击对应的 {n_points} 个点，完成后按任意键确认")
    cv2.imshow(f"[{seq_name}] Last Frame - click {n_points} points, press any key", last_frame)
    cv2.setMouseCallback(f"[{seq_name}] Last Frame - click {n_points} points, press any key",
                         click_event, points["fLast"])
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    print(f"  最后帧标注完成，共 {len(points['fLast'])} 个点")

    # ── 验证 & 保存 ───────────────────────────────────
    if len(points["f0"]) != len(points["fLast"]):
        print(f"  [警告] 点数不一致：第一帧 {len(points['f0'])} 个，最后帧 {len(points['fLast'])} 个，已跳过保存")
        return

    with open(ann_path, "w") as f:
        json.dump(points, f, indent=2)
    print(f"  标注已保存: {ann_path}")

# ── 批量标注 ──────────────────────────────────────────
for video_path in video_files:
    ann_path = Path(ANN_DIR) / f"annotation_{video_path.stem}.json"
    annotate_video(video_path, ann_path, n_points=N_POINTS)

print(f"\n{'='*50}")
print("全部标注完成！")

共找到 5 个视频:
  stable_seq_000_part_0_dif_0.mp4
  stable_seq_000_part_1_dif_0.mp4
  stable_seq_000_part_2_dif_0.mp4
  stable_seq_001_part_1_dif_0.mp4
  stable_seq_001_part_2_dif_0.mp4

标注: stable_seq_000_part_0_dif_0
  共 100 帧，请标注第0帧和第99帧

  >>> 第一帧：请点击 5 个点，完成后按任意键确认
  点 1: (271, 123)
  点 2: (351, 172)
  点 3: (188, 160)
  点 4: (374, 266)
  点 5: (229, 292)
  第一帧标注完成，共 5 个点

  >>> 最后一帧：请点击对应的 5 个点，完成后按任意键确认
  点 1: (243, 81)
  点 2: (372, 151)
  点 3: (114, 129)
  点 4: (402, 301)
  点 5: (205, 291)
  最后帧标注完成，共 5 个点
  标注已保存: D:\EndoTracker\vis_output\annotation\annotation_stable_seq_000_part_0_dif_0.json

标注: stable_seq_000_part_1_dif_0
  共 100 帧，请标注第0帧和第99帧

  >>> 第一帧：请点击 5 个点，完成后按任意键确认
  点 1: (60, 278)
  点 2: (134, 245)
  点 3: (107, 271)
  点 4: (86, 275)
  点 5: (123, 256)
  第一帧标注完成，共 5 个点

  >>> 最后一帧：请点击对应的 5 个点，完成后按任意键确认
  点 1: (68, 264)
  点 2: (247, 283)
  点 3: (330, 107)
  点 4: (245, 103)
  点 5: (92, 220)
  最后帧标注完成，共 5 个点
  标注已保存: D:\EndoTracker\vis_output\annotation\annotation_stable_seq_

In [ ]:
import sys, os, cv2, json, numpy as np, torch
from pathlib import Path
sys.path.insert(0, r'D:\EndoTracker\alltracker')

import utils.basic

from nets.alltracker import Net as AllTracker

# ── 配置 ──────────────────────────────────────────────

OUTPUT_DIR = r'D:\EndoTracker\vis_output'


MAX_FRAMES  = 100
os.makedirs(OUTPUT_DIR, exist_ok=True)




device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── 加载模型（只加载一次）────────────────────────────
model = AllTracker(seqlen=16).to(device)
ckpt  = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()
print("✅ 模型加载完成")

# ── 颜色 ──────────────────────────────────────────────
COLORS = [(0,0,255),(0,255,0),(255,0,0),(0,255,255),(255,0,255)]

# ── 批量处理 ──────────────────────────────────────────
video_files = sorted(Path(VIDEO_DIR).glob("*.mp4"))
print(f"共找到 {len(video_files)} 个视频\n")

all_results = {}

for video_path in video_files:
    seq_name = video_path.stem
    ann_path = Path(ANN_DIR) / f"annotation_{seq_name}.json"

    if not ann_path.exists():
        print(f"[跳过] {seq_name}，无标注文件")
        continue

    print(f"{'='*50}\n处理: {seq_name}")

    # 读视频
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"  [错误] 无法打开视频"); continue
    frames = []
    while len(frames) < MAX_FRAMES:
        ret, frame = cap.read()
        if not ret: break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    H, W = frames[0].shape[:2]
    T = len(frames)
    print(f"  帧数={T}, 分辨率={H}x{W}")

    video_tensor = torch.tensor(np.array(frames)).permute(0,3,1,2).unsqueeze(0).float().to(device)

    # 读标注
    with open(ann_path) as f:
        ann = json.load(f)
    f0_pts  = np.array(ann["f0"])
    gt_last = np.array(ann["fLast"])
    N = len(f0_pts)

    # ── 官方 forward_sliding 推理 ────────────────────
    grid_xy = utils.basic.gridcloud2d(1, H, W, norm=False, device=device).float()
    grid_xy = grid_xy.permute(0,2,1).reshape(1,1,2,H,W)

    with torch.no_grad():
        flows_e, visconf_maps_e, _, _ = model.forward_sliding(
            video_tensor, iters=4, sw=None, is_training=False
        )

    traj_maps = flows_e.to(device) + grid_xy   # [1,T,2,H,W]
    conf_maps = visconf_maps_e[:,:,1]           # [1,T,H,W]

    # ── 采样5个标注点 ─────────────────────────────────
    all_trajs = np.zeros((T, N, 2))
    all_confs = np.zeros((T, N))
    for i, (x0, y0) in enumerate(f0_pts):
        xi = min(max(int(round(x0)), 0), W-1)
        yi = min(max(int(round(y0)), 0), H-1)
        all_trajs[:, i, 0] = traj_maps[0, :, 0, yi, xi].cpu().numpy()
        all_trajs[:, i, 1] = traj_maps[0, :, 1, yi, xi].cpu().numpy()
        all_confs[:, i]    = conf_maps[0, :, yi, xi].cpu().numpy()

    # ── 计算指标 ──────────────────────────────────────
    pred_last = all_trajs[-1]
    errors = np.linalg.norm(pred_last - gt_last, axis=1)
    ATE = errors.mean()
    for i in range(N):
        print(f"  P{i+1}: 误差={errors[i]:.1f}px  conf={all_confs[-1,i]:.2f}")
    print(f"  平均ATE: {ATE:.1f}px")

    # ── 保存可视化视频 ────────────────────────────────
    out_video = str(Path(OUTPUT_DIR) / f"tracking_{seq_name}.mp4")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(out_video, fourcc, 10, (W, H))

    for t in range(T):
        frame = video_tensor[0,t].permute(1,2,0).cpu().numpy().astype(np.uint8)
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

        for i in range(N):
            color = COLORS[i % len(COLORS)]
            px = int(round(all_trajs[t, i, 0]))
            py = int(round(all_trajs[t, i, 1]))

            # 轨迹线
            trail = 20
            for t2 in range(max(0, t-trail), t):
                px2 = int(round(all_trajs[t2, i, 0]))
                py2 = int(round(all_trajs[t2, i, 1]))
                if 0<=px2<W and 0<=py2<H and 0<=px<W and 0<=py<H:
                    a = (t2-(t-trail))/trail
                    faded = tuple(int(c*a) for c in color)
                    cv2.line(frame, (px2,py2), (px,py), faded, 1)

            if 0<=px<W and 0<=py<H:
                cv2.circle(frame, (px,py), 6, color, -1)
                cv2.putText(frame, f"P{i+1}", (px+8,py-6),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

            # 最后一帧画 GT（白色叉）
            if t == T-1:
                gx = int(round(gt_last[i,0]))
                gy = int(round(gt_last[i,1]))
                cv2.drawMarker(frame,(gx,gy),(255,255,255),cv2.MARKER_CROSS,15,2)
                cv2.putText(frame, f"GT{i+1}", (gx+8,gy-6),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,255), 1)

        cv2.putText(frame, f"{seq_name}  Frame {t+1}/{T}", (10,25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)
        writer.write(frame)
    writer.release()
        # ── 最后一帧对比图 ────────────────────────────────
    last_frame = video_tensor[0, -1].permute(1,2,0).cpu().numpy().astype(np.uint8)
    last_frame = cv2.cvtColor(last_frame, cv2.COLOR_RGB2BGR)

    for i in range(N):
        color = COLORS[i % len(COLORS)]

        # 预测位置（实心圆）
        px = int(round(pred_last[i, 0]))
        py = int(round(pred_last[i, 1]))
        if 0 <= px < W and 0 <= py < H:
            cv2.circle(last_frame, (px, py), 8, color, -1)
            cv2.putText(last_frame, f"P{i+1}", (px+10, py-8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

        # GT位置（白色叉+圆圈）
        gx = int(round(gt_last[i, 0]))
        gy = int(round(gt_last[i, 1]))
        cv2.drawMarker(last_frame, (gx, gy), (255,255,255), cv2.MARKER_CROSS, 20, 2)
        cv2.circle(last_frame, (gx, gy), 8, (255,255,255), 2)
        cv2.putText(last_frame, f"GT{i+1}", (gx+10, gy+15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)

        # 连线（预测→GT，显示误差距离）
        if 0 <= px < W and 0 <= py < H:
            cv2.line(last_frame, (px,py), (gx,gy), (200,200,200), 1, cv2.LINE_AA)
            # 误差数值标在连线中点
            mx, my = (px+gx)//2, (py+gy)//2
            cv2.putText(last_frame, f"{errors[i]:.1f}px", (mx+4, my),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200,200,200), 1)

    # 标注图例
    cv2.putText(last_frame, "Pred: filled circle", (10, H-50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200,200,200), 1)
    cv2.putText(last_frame, "GT:   white cross+circle", (10, H-30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1)
    cv2.putText(last_frame, f"ATE={ATE:.1f}px", (10, H-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 1)

    out_img = str(Path(OUTPUT_DIR) / f"lastframe_{seq_name}.png")
    cv2.imwrite(out_img, last_frame)
    print(f"  ✅ 对比图 → {out_img}")
    print(f"  ✅ 视频 → {out_video}")

    # 保存 json
    result = {"seq": seq_name, "ATE": float(ATE),
              "per_point_error": errors.tolist(),
              "pred_last": pred_last.tolist(), "gt_last": gt_last.tolist()}
    all_results[seq_name] = result
    with open(Path(OUTPUT_DIR) / f"result_{seq_name}.json", "w") as f:
        json.dump(result, f, indent=2)

# ── 汇总 ──────────────────────────────────────────────
print(f"\n{'='*50}\n汇总:")
ATEs = [r['ATE'] for r in all_results.values()]
for seq, res in all_results.items():
    print(f"  {seq}: ATE={res['ATE']:.1f}px")
print(f"\n全部序列平均ATE: {np.mean(ATEs):.1f}px")

with open(Path(OUTPUT_DIR) / "summary.json", "w") as f:
    json.dump({"sequences": all_results, "mean_ATE": float(np.mean(ATEs))}, f, indent=2)
print(f"结果保存至 {OUTPUT_DIR}")

✅ 模型加载完成
共找到 5 个视频

处理: stable_seq_000_part_0_dif_0
  帧数=100, 分辨率=480x480


d:\Anaconda\envs\alltrack\lib\site-packages\torch\functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4383.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


  P1: 误差=15.6px  conf=0.90
  P2: 误差=1.0px  conf=0.91
  P3: 误差=4.5px  conf=0.75
  P4: 误差=14.6px  conf=0.71
  P5: 误差=64.3px  conf=0.84
  平均ATE: 20.0px
  ✅ 视频 → D:\EndoTracker\vis_output\tracking_stable_seq_000_part_0_dif_0.mp4
处理: stable_seq_000_part_1_dif_0
  帧数=100, 分辨率=480x480
  P1: 误差=19.7px  conf=0.08
  P2: 误差=85.0px  conf=0.06
  P3: 误差=242.9px  conf=0.10
  P4: 误差=208.4px  conf=0.09
  P5: 误差=75.3px  conf=0.08
  平均ATE: 126.3px
  ✅ 视频 → D:\EndoTracker\vis_output\tracking_stable_seq_000_part_1_dif_0.mp4
处理: stable_seq_000_part_2_dif_0
  帧数=100, 分辨率=480x480
  P1: 误差=327.9px  conf=0.71
  P2: 误差=47.5px  conf=0.51
  P3: 误差=230.8px  conf=0.26
  P4: 误差=206.1px  conf=0.84
  P5: 误差=270.5px  conf=0.76
  平均ATE: 216.6px
  ✅ 视频 → D:\EndoTracker\vis_output\tracking_stable_seq_000_part_2_dif_0.mp4
处理: stable_seq_001_part_1_dif_0
  帧数=100, 分辨率=480x480
  P1: 误差=17.1px  conf=0.43
  P2: 误差=52.9px  conf=0.92
  P3: 误差=28.0px  conf=0.97
  P4: 误差=29.5px  conf=0.96
  P5: 误差=50.0px  conf=0.90
  平均ATE: 35.5px
